In [1]:
import numpy as np
import matplotlib.pyplot as plt
import dask.dataframe as dd
import pandas as pd

%matplotlib inline

### Functions to run dask operations

In [2]:
def writedir(directory,overwrite=False):
    if overwrite:
        if os.path.exists(directory):
            shutil.rmtree(directory)
        os.makedirs(directory)
    else:
        if not os.path.exists(directory):
            os.makedirs(directory)

import os
import shutil
import dask
import time
import h5py
import threading
import dask.distributed

from time import sleep
from dask.distributed import Client,progress
from dask.distributed.diagnostics.plugin import WorkerPlugin
from dask_jobqueue import SLURMCluster

# from IPython.core.display import display, HTML
from IPython.display import display, HTML

## Hacky memory trim
import ctypes
import gc

def trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

class dask_controller: #adapted from Charles' code
    def __init__(self,n_workers=6,n_workers_min=6,local=True,queue="short",death_timeout=60.,\
                 walltime='01:00:00',cores=1,processes=1,memory='6GB',\
                 working_directory="./",job_extra_directives=[]):
        self.local = local
        self.n_workers = n_workers
        self.n_workers_min = n_workers_min
        self.walltime = walltime
        self.queue = queue
        self.death_timeout = death_timeout
        self.processes = processes
        self.memory = memory
        self.cores = cores
        self.working_directory = working_directory
        self.job_extra_directives = job_extra_directives

        self.futures = {}

        split_walltime = walltime.split(":")
        wall_hrs,wall_mins,wall_secs = tuple(split_walltime)
        ttl_wall_mins = (60*int(wall_hrs)) + int(wall_mins)

        if ttl_wall_mins < 10:
            raise ValueError("Walltime must be at least 10 mins long!")

        self.lifetime = str(max(ttl_wall_mins-10,10)) + "m"
        print(self.lifetime)
        print(walltime)
        self.worker_extra_args = ["--lifetime", self.lifetime, "--lifetime-stagger", "5m"]

        writedir(working_directory,overwrite=False)

    def startdask(self):
        if self.local:
            self.daskclient = Client()
            self.daskclient.cluster.scale(self.n_workers)
        else:
            self.daskcluster = SLURMCluster(n_workers=self.n_workers_min,queue=self.queue,death_timeout=self.death_timeout,walltime=self.walltime,\
                                   processes=self.processes,memory=self.memory,cores=self.cores,local_directory=self.working_directory,\
                                log_directory=self.working_directory,worker_extra_args=self.worker_extra_args,job_extra_directives=self.job_extra_directives)
            self.daskcluster.adapt(minimum=self.n_workers_min, maximum=self.n_workers,\
                                   interval="1m",wait_count=10)
            self.daskclient = Client(self.daskcluster)

    def shutdown(self, delete_files=True):
        self.reset_worker_memory()
        if not self.local:
            self.daskcluster.close()
        if delete_files:
            for item in os.listdir(self.working_directory):
                if "worker-" in item or "slurm-" in item or ".lock" in item:
                    path = "./" + item
                    if os.path.isfile(path):
                        os.remove(path)
                    elif os.path.isdir(path):
                        shutil.rmtree(path)

    def printprogress(self):
        complete = len([item for item in self.futures if item.status=="finished"])
        print(str(complete) + "/" + str(len(self.futures)))

    def displaydashboard(self):
        link = self.daskcluster.dashboard_link
        display(HTML('<a href="' + link +'">Dashboard</a>'))

    def mapfovs(self,function,fov_list,retries=0):
        self.function = function
        self.retries = retries
        def mapallfovs(fov_number,function=function):
            function(fov_number)
        self.futures = {}
        for fov in fov_list:
            future = self.daskclient.submit(mapallfovs,fov,retries=retries)
            self.futures[fov] = future

    def retry_failed(self):
        self.failed_fovs = [fov for fov,future in self.futures.items() if future.status != 'finished']
        out = self.daskclient.restart()
        self.mapfovs(self.function,self.failed_fovs,retries=self.retries)

    def retry_processing(self):
        self.proc_fovs = [fov for fov,future in self.futures.items() if future.status == 'pending']
        out = self.daskclient.restart()
        self.mapfovs(self.function,self.proc_fovs,retries=self.retries)

    def restart(self):
        self.daskclient.restart()
        with self.failure_lock:
            self.failure_counter_variable.set(0)

    def reset_worker_memory(self):
        self.daskclient.cancel([val for key,val in self.futures.items()])
        self.daskclient.run(gc.collect)
        self.daskclient.run(trim_memory)
        print("Done.")

class hdf5lock:
    def __init__(self,filepath,updateperiod=0.1):
        self.filepath = filepath
        self.lockfile = filepath + ".lock"
        self.updateperiod = updateperiod

    def _lock(self):
        while True:
            if not os.path.exists(self.lockfile):
                open(self.lockfile,'w').close()
                break
            sleep(self.updateperiod)

    def _apply_fn(self,function,iomode,*args,**kwargs):
        try:
            fn_output = function(self.filepath,iomode,*args,**kwargs)
            os.remove(self.lockfile)
            return fn_output
        except:
            os.remove(self.lockfile)
            raise

    def lockedfn(self,function,iomode,*args,**kwargs):
        self._lock()
        fn_output = self._apply_fn(function,iomode,*args,**kwargs)
        return fn_output

def transferjob(sourcedir,targetdir,single_file=False):
    mkdircmd = "mkdir -p '" + targetdir + "'"
    if single_file:
        rsynccmd = "rsync -r '" + sourcedir + "' '" + targetdir + "'"
    else:
        rsynccmd = "rsync -r '" + sourcedir + "/' '" + targetdir + "'"
    wrapcmd = mkdircmd + " && " + rsynccmd
    cmd = "sbatch -p transfer -t 0-12:00 --wrap=\"" + wrapcmd + "\""
    print(cmd)
    os.system(cmd)


In [3]:
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

dask_wd = "/home/lag36/scratch/lag36/dask"

dask_controller_instance = dask_controller(
    walltime="3:00:00",
    local=False,
    n_workers=10,
    n_workers_min=10,#200,
    memory="20GB",
    working_directory=dask_wd,
)
dask_controller_instance.startdask()
dask_controller_instance.displaydashboard()

170m
3:00:00


In [ ]:
# basepath = "/home/lag36/scratch/lag36/2024-12-12_lLAG8_EZRDM_Run-02_3-Fiducials_Auto-Media-Switch/Growth_Division/2024-12-12_lLAG8-EZ-30"

# df_obs_growth = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Lineage_Growth_Observations",calculate_divisions=True)
# df_obs_obs = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Lineage_Observations",calculate_divisions=True)
# df_obs_cell_cycle = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Lineage_Cell_Cycle",calculate_divisions=True)
df_bar = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Final_Barcodes_df", engine="pyarrow",calculate_divisions=True)

# Get also lineage information
# df_lineage = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/lineage", engine="pyarrow",calculate_divisions=True)
# df = pd.read_hdf("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Barcodes/barcode_output_df.hdf5")
bc_kymo_metadata = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Barcodes/kymograph/metadata", calculate_divisions=True)


### Understanding how the tabular data is structured

- `df_obs_growth`, `df_obs_obs`, `df_obs_cell_cycle`, and `df_bar` all are essentially indexed by `phenotype trenchid`.

- `df_bar` has all the barcode information, and the mapping to the phenotype kymographs. In this table:
    - `phenotype trenchid` is the trench index from the phenotyping data. This can be used to map to the lineage dataframe.
    - `trenchid` is the trench index from the FISH data. It can be used to map to the FISH kymograph metadata to obtain the file indices and file trench indices.
    - `File Index`, `File Trench Index` are the indices for the **phenotype** kymographs.
    - `fov`, `timepoints`, `y (local)`, `x (local)` also correspond to the **phenotype** kymographs.
    - `Barcode` is the barcode sequence from the phenotype, which is found from the FISH images. It is arranged by channel, and then by cycle number. That is: $(\text{ch}_1, \text{cyc}_1), (\text{ch}_1, \text{cyc}_2), ..., (\text{ch}_1, \text{cyc}_{15}), (\text{ch}_2, \text{cyc}_1),..., (\text{ch}_2, \text{cyc}_{15})$.
    - `barcode` is the barcode sequence inferred from Nanopore sequencing. It is arranged by bit number in the actual RNA barcode. There is a lookup table that maps positions in barcode RNA to cycle order. In my case it is:

    ```
    cycleorder = [0,1,6,7,12,13,18,19,24,25,4,5,10,11,16,
                  2,3,8,9,14,15,20,21,26,27,17,22,23,28,29],
    ```


### Load, filter, and group tabular data

In [18]:
# Generate metadata df from parquet
headpath = '/home/lag36/scratch/lag36/2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed/'
suffix = '/Barcodes/kymograph/metadata'

cols_bar_kymo_meta = ['fov', 'timepoints', 'lane orientation', 'y (local)', 
                      'x (local)', 'row', 'File Index', 'File Trench Index',
                      'trenchid', 'Trenchid Timepoint Index']

df_bar_kymo_meta = (dd.read_parquet(headpath+suffix,
                         engine="pyarrow",
                         calculate_divisions=True,
                         columns = cols_bar_kymo_meta
                         )
                         .astype({'fov':'int16', 'timepoints':'int16', 'lane orientation':'category', 'row':'int8',
                                            'File Index':'int32', 'File Trench Index':'int32', 'trenchid': 'int32'})
                         .compute()
                        
)
df_bar_kymo_meta.to_hdf(headpath+"/Barcodes/kymograph_metadata_df.hdf5", key='data', format='table')

In [ ]:
# headpath = '/home/lag36/scratch/lag36/2024-12-12_lLAG8_EZRDM_Run-02_3-Fiducials_Auto-Media-Switch/'
headpath = '/home/lag36/scratch/lag36/2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed/'

# Here trenchid works the same as above, what we don't have yet is phenotype trenchid
# df_bar = pd.read_hdf(headpath+'/Barcodes/barcode_output_df_fiducials_hamming-0_rm-1-28.hdf5') # From the end of notebook 1 (No mapping yet)

cols_bar = ['phenotype trenchid', 'fov', 'timepoints', 'lane orientation',
            'y (local)', 'x (local)', 'row', 'File Index', 
            'File Trench Index', 'trenchid', 'Barcode', 'barcode', 'File Parquet Index', 'Trenchid Timepoint Index']

# suffix = '/Growth_Division/2024-12-12_lLAG8-EZ-30_Final_Barcodes_df'
suffix = '/Growth_Division/2024-12-05_lLAG10-MBM-37_Run-02_Final_Barcodes_df'
df_bar = (dd.read_parquet(headpath+suffix,
                         engine="pyarrow",
                         calculate_divisions=True,
                         columns = cols_bar
                         )
          .astype({'phenotype trenchid':'int32', 'fov':'int16', 'timepoints':'int16', 'lane orientation':'category',
                   'row':'int8', 'File Index':'int32', 'File Trench Index':'int32', 'trenchid': 'int32'})
          .compute()
         )# From the end of nb 3 (already mapped)


cols_bar_kymo_meta = ['fov', 'timepoints', 'lane orientation', 'y (local)', 
                      'x (local)', 'row', 'File Index', 'File Trench Index',
                      'trenchid', 'Trenchid Timepoint Index']

cols_bar_fovs_meta = ['t', 'File Index', 'Image Index']


df_bar_kymo_meta = (pd.read_hdf(headpath+"/Barcodes/kymograph_metadata_df.hdf5",
                                key='data',
                               )
                    .loc[:, cols_bar_kymo_meta] # Typically dask dataframe but I saved it as hdf5
                    .astype({'fov':'int16', 'timepoints':'int16', 'lane orientation':'category', 'row':'int8',
                                            'File Index':'int32', 'File Trench Index':'int32', 'trenchid': 'int32'})
                    # Merge Barcode FOV metadata
                    .merge(
                        (pd.read_hdf(headpath+"/Barcodes/metadata.hdf5")
                         .loc[:, cols_bar_fovs_meta]
                         .astype({'File Index':'int32', 'Image Index':'int32'})
                         # Better way to do this?
                         .reset_index()
                         .astype({'fov':'int16', 'timepoints':'int16'})
                        ),
                        on=['fov', 'timepoints'],
                        how='left',
                        suffixes=('', '_fov') 
                    )
                   )

In [203]:
# Get the df right after FISH decoding (i.e. end of notebook 1. No mapping to phenotypes)
# NOTE: It does not have the next closest barcode!!

# For lLAG8_EZRDM_Run-02_3-Fiducials_Auto-Media-Switch
# suffix = '/Barcodes/barcode_output_df_fiducials_hamming-0_rm-1-28.hdf5'
# For lLAG10-MBM
suffix = 'Barcodes/barcode_output_df_fiducials_hamming-2_rmBits-None.hdf5'

# df_bar_fish = pd.read_hdf(headpath+suffix).sort_values(by='trenchid')
df_bar_fish = pd.read_hdf(headpath+suffix)
df_bar_fish

with pd.HDFStore(headpath+suffix,'r') as store:
    df_bar_fish_metadata = store.get_storer('barcodes').attrs.metadata

In [57]:
# Phenotype data
# df_phe apparently not needed, save perhaps the Trench Score
# cols_phe = ['File Index', 'File Trench Index', 'timepoints', 'fov', 
#             'row', 'trenchid', 'lane orientation', 'Trench Score',
#             'Trenchid Timepoint Index']
# df_phe = dd.read_parquet(headpath+"/Growth_Division/lineage/output",
#                          engine="pyarrow",
#                          calculate_divisions=True,
#                          columns=cols_phe,
#                         ) # Here trenchid maps to the phenotype kymo meta

# Apparently not needed either
# cols_phe_kymo_meta = ['fov', 'timepoints', 'lane orientation', 'y (local)', 
#                       'x (local)', 'row', 'File Index', 'File Trench Index',
#                       'trenchid']
# df_phe_kymo_meta = dd.read_parquet(headpath+"/Growth_Division/kymograph/metadata",
#                                    engine="pyarrow",
#                                    calculate_divisions=True,
#                                    columns=cols_phe_kymo_meta,
#                                   )

cols_phe_fovs_meta = ['t', 'File Index', 'Image Index']
df_phe_fovs_meta = (pd.read_hdf(headpath+"/Growth_Division/metadata.hdf5")
                    .loc[:, cols_phe_fovs_meta]
                    .astype({'File Index':'int32', 'Image Index':'int32'})
                    #  # Better way to do this?
                    .reset_index()
                    .astype({'fov':'int16', 'timepoints':'int16'})
                   )

df_bar = df_bar.merge(df_phe_fovs_meta,
                      on=['fov', 'timepoints'],
                      how='left',
                      suffixes=('', '_fov'))

In [58]:
# I should not assume that everything is in a single hdf5 file for a single trenchid across all timepoints
# df_bar[df_bar['phenotype trenchid'] == 1103657].tail(50)

### Get barcode kymographs, and format as xarray

In [23]:
# Get barcode images
import xarray as xr
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import holoviews as hv
from holoviews.operation.datashader import rasterize
import hvplot.xarray # noqa
import panel as pn
import fsspec

pn.extension('tabulator')
# hv.extension('bokeh')
hv.extension('bokeh', 'matplotlib')

In [ ]:
# Open kymograph hdf5 files into an xarray dataset
headpath_bar_kymo = headpath + '/Barcodes/kymograph/kymograph_'
# This seems to be assuming that all timepoints are in the same file for a given trench
df_bar_kymo_lookup = df_bar_kymo_meta[['trenchid', 'File Index', 'File Trench Index']].groupby('trenchid').first().reset_index()

#
file_index = 100
data_kymo = (xr
        .open_dataset(headpath_bar_kymo+str(file_index)+'.hdf5', phony_dims='sort')
        .swap_dims({'phony_dim_0':'File Trench Index', 'phony_dim_1':'timepoints', 'phony_dim_2':'y', 'phony_dim_3':'x' })
)

data_kymo = (data_kymo
             .assign_coords({'File Trench Index': np.arange(data_kymo.sizes['File Trench Index'], dtype=np.int16),
                             'timepoints': np.arange(data_kymo.sizes['timepoints'], dtype=np.int16),
                             'y': np.arange(data_kymo.sizes['y'], dtype=np.int16),
                             'x': np.arange(data_kymo.sizes['x'], dtype=np.int16),
                             })
                             .assign_coords(trenchid= ('File Trench Index', df_bar_kymo_lookup.loc[df_bar_kymo_lookup['File Index']==file_index, 'trenchid']))
)

In [81]:
# Are there nan values in data_kymo
data_kymo

<xarray.Dataset> Size: 99MB
Dimensions:            (File Trench Index: 666, timepoints: 15, y: 150, x: 11)
Coordinates:
  * File Trench Index  (File Trench Index) int16 1kB 0 1 2 3 ... 662 663 664 665
  * timepoints         (timepoints) int16 30B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14
  * y                  (y) int16 300B 0 1 2 3 4 5 6 ... 144 145 146 147 148 149
  * x                  (x) int16 22B 0 1 2 3 4 5 6 7 8 9 10
    trenchid           (File Trench Index) int32 3kB 66600 66601 ... 67264 67265
Data variables:
    BF                 (File Trench Index, timepoints, y, x) uint16 33MB ...
    Cy5                (File Trench Index, timepoints, y, x) uint16 33MB ...
    RFP                (File Trench Index, timepoints, y, x) uint16 33MB ...

#### Plot movies for a given kymograph

In [11]:
def plot_bc_trench_per_trenchid_timepoint(data_kymo):
    '''
    Plot the kymograph for each trenchid and timepoint
    It has scrollers for File Trench Index and timepoints
    data_kymo: xarray dataset
    '''
    # Generate it for 3 channels: RFP, GFP, and YFP
    hvplot_app = {}
    for channel in ['BF', 'RFP', 'Cy5']:
        hvplot_app[channel] = (data_kymo[channel]
                               .hvplot.image(
                                   groupby=['File Trench Index', 'timepoints'],
                                   title=channel,
                                   cmap='gray',
                                   data_aspect=1,
                                   # widget_location='bottom',
                                #    rasterize=True,
                                )
                               .opts(invert_yaxis=True,
                                     # Remove axes labels and ticks
                                     xaxis=None, yaxis=None,
                                     colorbar=False,
                                    #  height=300
                                     ))
    return hvplot_app['BF'] + hvplot_app['Cy5'] + hvplot_app['RFP']

# plot_bc_trench_per_trenchid_timepoint(data_kymo)

### Show barcode annotation for all trenches in an hdf5 file

In [83]:
def map_bc_to_pheno(barcode,
                    cycleorder = [0,1,6,7,12,13,18,19,24,25,4,5,10,11,16,
                                  2,3,8,9,14,15,20,21,26,27,17,22,23,28,29],
                    ):
    barcode_phe = [-1]*30
    for i, bit in enumerate(barcode):
        # print(i, bit)
        # Find the position of i in cycleorder
        i_phe = cycleorder.index(i)
        barcode_phe[i_phe] = bit
    barcode_phe = ''.join(barcode_phe)
    return barcode_phe

In [84]:
df_bar_annotations = \
    (df_bar
     .groupby('trenchid').first()[['Barcode', 'barcode']]
     .assign(barcode_phe = lambda df_: df_['barcode'].apply(map_bc_to_pheno))
    )

# Generate a dataframe of bits per timepoint
n_bits = 30
n_channels = 2
channels = {0:'RFP', 1:'Cy5'}
df_bar_annotations = \
     (df_bar_annotations
      # map list only to the columns 'Barcode' and 'barcode'
     .map(list)
     .explode(column=['Barcode','barcode', 'barcode_phe'])
     .astype({'Barcode':np.int8, 'barcode':np.int8, 'barcode_phe':np.int8})
     .reset_index()
     # Assign a column of timepoints from 0 to 30 for a each trenchid
     .assign(order_in_barcode=lambda df_: df_.groupby('trenchid').cumcount())
     .assign(channel = lambda df_: df_['order_in_barcode'] // (n_bits//n_channels)) # The parenthesis is necessary for integer division
     .assign(timepoints = lambda df_: df_['order_in_barcode'] % (n_bits//n_channels))
     .astype({'channel':'category', 'timepoints':'int16'})
     .assign(channel = lambda df_: df_['channel'].map(channels))
     # Pivot channels to columns
     .pivot(index = ['trenchid', 'timepoints'], columns='channel', values=['Barcode', 'barcode', 'barcode_phe'])
     .astype({'Barcode':'category', 'barcode':'category', 'barcode_phe':'category'})
     # .swaplevel(axis=1)
     # Merge both column index
)
df_bar_annotations.columns = df_bar_annotations.columns.map('{0[0]}_{0[1]}'.format)

In [87]:
# Find lookup on kymo_metadata of the ones that were mapped to a phenotype trench
df_bar_kymo_lookup_filt = df_bar_kymo_lookup[df_bar_kymo_lookup['trenchid'].isin(df_bar_annotations.index.get_level_values(0).unique())]

In [89]:
df_bar_kymo_lookup_filt

,trenchid,File Index,File Trench Index
3,3,0,3
22,22,0,22
28,28,0,28
146,146,0,146
181,181,0,181
...,...,...,...
1363488,1363488,2047,186
1363495,1363495,2047,193
1363497,1363497,2047,195
1363499,1363499,2047,197


In [90]:
cols_bar_annotations = ['Barcode_RFP', 'barcode_phe_RFP', 
                        'Barcode_Cy5', 'barcode_phe_Cy5']

In [91]:
# df_lookup_files = df_bar[['trenchid', 'File Index', 'File Trench Index']].groupby('trenchid').first().reset_index() # There must be a better way!
bar_annotations = xr.Dataset.from_dataframe(df_bar_annotations[cols_bar_annotations])
bar_annotations = bar_annotations.assign_coords(x=('timepoints', [5]*bar_annotations.sizes['timepoints']), 
                                                y=('timepoints',[20]*bar_annotations.sizes['timepoints']),
                                                # This assumes one-one mapping, but there is no direct lookup, it might be prone to errors
                                                file_index=('trenchid', df_bar_kymo_lookup_filt['File Index']),
                                                file_trench_index=('trenchid', df_bar_kymo_lookup_filt['File Trench Index'])
                                                )
bar_annotations

<xarray.Dataset> Size: 13MB
Dimensions:            (trenchid: 183502, timepoints: 15)
Coordinates:
  * trenchid           (trenchid) int32 734kB 3 22 28 ... 1363499 1363516
  * timepoints         (timepoints) int16 30B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    x                  (timepoints) int64 120B 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5
    y                  (timepoints) int64 120B 20 20 20 20 20 ... 20 20 20 20 20
    file_index         (trenchid) int32 734kB 0 0 0 0 0 ... 2047 2047 2047 2047
    file_trench_index  (trenchid) int32 734kB 3 22 28 146 ... 193 195 197 214
Data variables:
    Barcode_RFP        (trenchid, timepoints) int8 3MB 1 1 1 1 1 1 ... 0 1 0 0 0
    barcode_phe_RFP    (trenchid, timepoints) int8 3MB 1 1 1 1 1 1 ... 0 1 0 0 0
    Barcode_Cy5        (trenchid, timepoints) int8 3MB 1 0 1 0 0 0 ... 0 0 1 1 1
    barcode_phe_Cy5    (trenchid, timepoints) int8 3MB 1 0 0 0 0 0 ... 0 0 1 1 1

In [92]:
df_bar_kymo_lookup_filt_in_file = df_bar_kymo_lookup_filt[df_bar_kymo_lookup_filt['File Index'] == file_index]
df_bar_kymo_lookup_filt_in_file

,trenchid,File Index,File Trench Index
66600,66600,100,0
66614,66614,100,14
66619,66619,100,19
66632,66632,100,32
66634,66634,100,34
...,...,...,...
67189,67189,100,589
67191,67191,100,591
67192,67192,100,592
67205,67205,100,605


In [34]:
# ATTEMPTS TO SHOW LABELS

# # It works when selecting a unique trenchid
# label_hv_RFP = (bar_annotations
#                 # Select first 2 indices
#                 .sel(trenchid=df_bar_kymo_lookup_filt_in_file.iloc[1]['trenchid'])
#                 .hvplot.labels(x='x', y='y', text='Cy5', groupby=['timepoints'])
# )
# label_hv_RFP

# What about 2 trenchids: DOES NOT WORK
# label_hv_RFP = (bar_annotations
#                 .sel(trenchid=slice(0, 100))
#                 .hvplot.labels(x='x', y='y', text='RFP', groupby=['trenchid','timepoints'])
# )
# label_hv_RFP

# A Holomap is possible but it is very slow
def label_barcode(trenchid, timepoint, channel):
    ann_single = bar_annotations.sel(trenchid=trenchid).isel(timepoints=timepoint)
    x, y, text = ann_single['x'].values, ann_single['y'].values, ann_single[channel].values
    labels = hv.Labels((x, y, text))
    return labels

channel='Cy5'
trenchid=57
timepoints=0

# Define mapping dictionary
labels_dict = {(trenchid, timepoint): label_barcode(trenchid, timepoint, channel) 
               for trenchid in df_bar_kymo_lookup_filt['trenchid'].values[:5]
               for timepoint in range(15)}

hv.HoloMap(labels_dict, kdims=['trenchid', 'timepoints'])

KeyError: "No variable named 'Cy5'. Variables on the dataset include ['Barcode_RFP', 'barcode_phe_RFP', 'Barcode_Cy5', 'barcode_phe_Cy5', 'trenchid', 'timepoints', 'x', 'y', 'file_index', 'file_trench_index']"

In [130]:
# Choose trenchid
trenchid = df_bar_kymo_lookup_filt_in_file.iloc[2]['trenchid']

# Generate labels
label_hv_RFP = bar_annotations.sel(trenchid=trenchid).hvplot.labels(x='x', y='y', text='RFP', groupby=['timepoints'])
label_hv_Cy5 = bar_annotations.sel(trenchid=trenchid).hvplot.labels(x='x', y='y', text='Cy5', groupby=['timepoints'])

# Generate it for 3 channels: RFP, GFP, and YFP
hvplot_app = {}
for channel in ['BF', 'RFP', 'Cy5']:
    hvplot_app[channel] = (data_kymo.where(data_kymo['trenchid']== trenchid, drop=True).isel({'File Trench Index':0})[channel]
                        #    .isel({'File Trench Index':57})
                           .hvplot.image(
        groupby=['timepoints'],
        title=channel,
        cmap='gray',
        # clim=(0, 20),
        data_aspect=1,
        label = str(trenchid),
        # widget_location='bottom',
        # rasterize=True  
    ).opts(invert_yaxis=True,
           # Remove axes labels and ticks
              xaxis=None, yaxis=None,
              # Remove colorbar
                colorbar=False,
           ))
# hvplot_app['BF']*label_hv_Cy5 + hvplot_app['Cy5']*label_hv_Cy5 + hvplot_app['RFP']*label_hv_RFP

In [ ]:
# Generate it for 3 channels: RFP, GFP, and YFP
hvplot_app = {}
for channel in ['BF', 'RFP', 'Cy5']:
    hvplot_app[channel] = (data_kymo[channel]
                        #    .isel({'File Trench Index':57})
                           .hvplot.image(
        groupby=['File Trench Index', 'timepoints'],
        title=channel,
        cmap='gray',
        # clim=(0, 20),
        data_aspect=1,
        # widget_location='bottom',
        rasterize=True  
    ).opts(invert_yaxis=True,
           # Remove axes labels and ticks
              xaxis=None, yaxis=None,
              # Remove colorbar
                colorbar=False,
           ))
# hvplot_app['BF'] + hvplot_app['Cy5'] + hvplot_app['RFP']

In [ ]:
hvplot_app['RFP']+ hvplot_app['Cy5']

### Show barcode annotation for a kymo

In [93]:
# Aggregate time axis over x, generating kymographs
data_kymos = xr.concat([data_kymo.isel({'timepoints':i}) for i in data_kymo['timepoints'].values], dim='x')
data_kymos = data_kymos.assign_coords({
    # 'File Trench Index': np.arange(data_kymos.sizes['File Trench Index'], dtype=np.int16),
                        # 'timepoints': np.arange(data.sizes['timepoints'], dtype=np.int16),
                        # 'y': np.arange(data_kymos.sizes['y'], dtype=np.int16),
                        'x': np.arange(data_kymos.sizes['x'], dtype=np.int16),})

data_kymos

<xarray.Dataset> Size: 99MB
Dimensions:            (File Trench Index: 666, y: 150, x: 165)
Coordinates:
  * File Trench Index  (File Trench Index) int16 1kB 0 1 2 3 ... 662 663 664 665
    timepoints         (x) int16 330B 0 0 0 0 0 0 0 0 ... 14 14 14 14 14 14 14
  * y                  (y) int16 300B 0 1 2 3 4 5 6 ... 144 145 146 147 148 149
    trenchid           (File Trench Index) int32 3kB 66600 66601 ... 67264 67265
  * x                  (x) int16 330B 0 1 2 3 4 5 6 ... 159 160 161 162 163 164
Data variables:
    BF                 (File Trench Index, y, x) uint16 33MB 13261 ... 3035
    Cy5                (File Trench Index, y, x) uint16 33MB 3229 3104 ... 2463
    RFP                (File Trench Index, y, x) uint16 33MB 1935 1752 ... 1750

In [94]:
trench_width = 11
number_of_cycles = 14 #15?
bar_annotations_kymo = xr.concat([bar_annotations.isel({'timepoints':i}) for i in bar_annotations['timepoints'].values], dim='x')
bar_annotations_kymo = (bar_annotations_kymo
                        .assign_coords({'x': np.arange(bar_annotations_kymo.sizes['x'], dtype=np.int16)*trench_width,} ) # Hacky? I should use timepoints
                        .transpose('trenchid', 'x')
)
# Transpose

In [45]:
bar_annotations_kymo

<xarray.Dataset> Size: 12MB
Dimensions:            (x: 15, trenchid: 167546)
Coordinates:
  * trenchid           (trenchid) int32 670kB 57 198 217 ... 1341280 1341284
    timepoints         (x) int16 30B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    y                  int64 8B 20
    file_index         (trenchid) int32 670kB 0 0 0 0 0 ... 2013 2013 2013 2013
    file_trench_index  (trenchid) int32 670kB 57 198 217 219 ... 559 577 622 626
  * x                  (x) int16 30B 0 11 22 33 44 55 ... 99 110 121 132 143 154
Data variables:
    Barcode_RFP        (trenchid, x) int8 3MB 1 0 1 1 1 1 1 1 ... 0 0 0 1 1 0 0
    barcode_phe_RFP    (trenchid, x) int8 3MB 1 0 1 1 1 1 1 1 ... 0 0 0 1 1 0 0
    Barcode_Cy5        (trenchid, x) int8 3MB 1 1 1 1 1 1 1 1 ... 1 0 0 1 1 0 1
    barcode_phe_Cy5    (trenchid, x) int8 3MB 1 1 1 1 1 1 1 1 ... 1 0 0 1 1 0 1

In [100]:
# It does not works when selecting a unique trenchid
# label_hv_RFP = (bar_annotations_kymo
#                 # Select first 2 indices
#                 # .sel(trenchid=trenchids_annotated.iloc[0]['trenchid'])
#                 .hvplot.labels(x='x', y='y', text='RFP', groupby=['trenchid'])
# )
# label_hv_RFP

In [37]:
def show_bc_kymo(trenchid, channel):
    # Generate it for 3 channels: RFP, GFP, and YFP
    label_hv = bar_annotations_kymo.sel(trenchid=trenchid).hvplot.labels(x='x', y='y', text=channel).opts(text_font_size='11pt', xoffset=trench_width//2+2, text_color='red')
    hvplot_app = data_kymos[channel].where(data_kymo['trenchid']== trenchid, drop=True).isel({'File Trench Index':0}).hvplot.image(
        # groupby='File Trench Index',
        title=channel,
        cmap='gray',
        # clim=(0, 20),
        data_aspect=1,
        # widget_location='bottom',
        rasterize=False,
        label = str(trenchid)
        # rtol=1e-2,
        # Make it have a single column

    ).opts(invert_yaxis=True,
            # Remove axes labels and ticks
                xaxis=None, yaxis=None,
                # Remove colorbar
                colorbar=False,
                # Single column
                # ncols=1
            )
    # return hvplot_app['BF'] + hvplot_app['RFP']*label_hv_RFP + hvplot_app['Cy5']*label_hv_Cy5
    return hvplot_app*label_hv

In [ ]:
bc_kymos_map_single = {trenchid: show_bc_kymo(trenchid, 'RFP') for trenchid in trenchids_annotated['trenchid'].values[:3]}

In [ ]:
def show_bc_kymo(trenchid, channel = 'RFP'):
    # Generate it for 3 channels: RFP, GFP, and YFP
    label_hv = bar_annotations_kymo.sel(trenchid=trenchid).hvplot.labels(x='x', y='y', text=channel).opts(text_font_size='11pt', xoffset=trench_width//2+2, text_color='red')
    hvplot_app = data_kymos[channel].where(data_kymo['trenchid']== trenchid, drop=True).isel({'File Trench Index':0}).hvplot.image(
        # groupby='File Trench Index',
        title=channel,
        cmap='gray',
        # clim=(0, 20),
        data_aspect=1,
        # widget_location='bottom',
        rasterize=False,
        label = str(trenchid)
        # rtol=1e-2,
        # Make it have a single column

    ).opts(invert_yaxis=True,
            # Remove axes labels and ticks
                xaxis=None, yaxis=None,
                # Remove colorbar
                colorbar=False,
                # Single column
                # ncols=1
            )
    # return hvplot_app['BF'] + hvplot_app['RFP']*label_hv_RFP + hvplot_app['Cy5']*label_hv_Cy5
    return hvplot_app*label_hv

In [ ]:
hv.DynamicMap(show_bc_kymo, kdims=['trenchid']).redim.values(trenchid=[57, 198])

In [136]:
bar_annotations_kymo

<xarray.Dataset> Size: 12MB
Dimensions:            (x: 15, trenchid: 167546)
Coordinates:
  * trenchid           (trenchid) int32 670kB 57 198 217 ... 1341280 1341284
    timepoints         (x) int16 30B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    y                  int64 8B 20
    file_index         (trenchid) int32 670kB 0 0 0 0 0 ... 2013 2013 2013 2013
    file_trench_index  (trenchid) int32 670kB 57 198 217 219 ... 559 577 622 626
  * x                  (x) int16 30B 0 11 22 33 44 55 ... 99 110 121 132 143 154
Data variables:
    Barcode_RFP        (trenchid, x) int8 3MB 1 0 1 1 1 1 1 1 ... 0 0 0 1 1 0 0
    barcode_phe_RFP    (trenchid, x) int8 3MB 1 0 1 1 1 1 1 1 ... 0 0 0 1 1 0 0
    Barcode_Cy5        (trenchid, x) int8 3MB 1 1 1 1 1 1 1 1 ... 1 0 0 1 1 0 1
    barcode_phe_Cy5    (trenchid, x) int8 3MB 1 1 1 1 1 1 1 1 ... 1 0 0 1 1 0 1

In [322]:
def show_bc_kymos(trenchid, show_labels=True):


    # Make hv.labels from 0 to 14
    # label_hv_t = bar_annotations_kymo.hvplot.labels(x='x', y='y', text='x').opts(text_font_size='10pt', xoffset=trench_width//2+1, yoffset=-15, text_color='white')
    label_hv_t = bar_annotations_kymo.isel(trenchid=0).hvplot.labels(x='x', y='y', text='timepoints').opts(text_font_size='10pt', xoffset=trench_width//2+1, yoffset=-15, text_color='white') # Hacky, correct later
    if show_labels:
        
        label_hv_RFP = bar_annotations_kymo.sel(trenchid=trenchid).hvplot.labels(x='x', y='y', text='Barcode_RFP').opts(text_font_size='11pt', xoffset=trench_width//2+1, text_color='red')
        label_hv_Cy5 = bar_annotations_kymo.sel(trenchid=trenchid).hvplot.labels(x='x', y='y', text='Barcode_Cy5').opts(text_font_size='11pt', xoffset=trench_width//2+1, text_color='red')
        label_hv_RFP_sequence = bar_annotations_kymo.sel(trenchid=trenchid).hvplot.labels(x='x', y='y', text='barcode_phe_RFP').opts(text_font_size='11pt', xoffset=trench_width//2+1, yoffset=15,text_color='yellow')
        label_hv_Cy5_sequence = bar_annotations_kymo.sel(trenchid=trenchid).hvplot.labels(x='x', y='y', text='barcode_phe_Cy5').opts(text_font_size='11pt', xoffset=trench_width//2+1, yoffset=15,text_color='yellow')
    hvplot_app = {}
    for channel in ['BF', 'RFP', 'Cy5']:
        hvplot_app[channel] = data_kymos[channel].where(data_kymo['trenchid']== trenchid, drop=True).isel({'File Trench Index':0}).hvplot.image(
            # groupby='File Trench Index',
            title=channel,
            cmap='gray',
            # clim=(0, 20),
            # data_aspect=1,
            # widget_location='bottom',
            # rasterize=True,
            # label = str(trenchid)
            # rtol=1e-2,
            # Make it have a single column
        ).opts(invert_yaxis=True,
                  xaxis=None, yaxis=None,
                    colorbar=False,
                    # Single column
                    # ncols=1,
                    padding=0.02,
                    margin=0,
                     border=0,
                     aspect='equal',
                     width=300
               )
    if show_labels:
        return (hvplot_app['BF']*label_hv_t + hvplot_app['RFP']*label_hv_RFP*label_hv_RFP_sequence*label_hv_t + hvplot_app['Cy5']*label_hv_Cy5*label_hv_Cy5_sequence*label_hv_t)
    else:
        return hvplot_app['BF']*label_hv_t + hvplot_app['RFP']*label_hv_t + hvplot_app['Cy5']*label_hv_t

In [96]:
df_bar_kymo_lookup_filt_in_file.head()

,trenchid,File Index,File Trench Index
66600,66600,100,0
66614,66614,100,14
66619,66619,100,19
66632,66632,100,32
66634,66634,100,34


In [323]:
show_bc_kymos(66600, show_labels=False)

:Layout
   .Overlay.I   :Overlay
      .Image.I  :Image   [x,y]   (BF)
      .Labels.I :Labels   [x,y]   (timepoints)
   .Overlay.II  :Overlay
      .Image.I  :Image   [x,y]   (RFP)
      .Labels.I :Labels   [x,y]   (timepoints)
   .Overlay.III :Overlay
      .Image.I  :Image   [x,y]   (Cy5)
      .Labels.I :Labels   [x,y]   (timepoints)

#### Percentiles!

In [ ]:
# 2024-12-12_lLAG8_EZRDM_Run-02_3-Fiducials_Auto-Media-Switch
# df_percentiles = dd.read_parquet(headpath+'/Barcodes/percentiles',engine="pyarrow",calculate_divisions=True)

# Convert from dd to df
# 2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed
df_percentiles = (dd.read_parquet('/home/lag36/scratch/lag36/2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed/Barcodes/percentiles',
                                 engine="pyarrow",
                                 calculate_divisions=True,
                                 columns=['timepoints', 'trenchid', 'RFP 98th Percentile', 'Cy5 98th Percentile'])
                    .astype({'trenchid':'int32', 'timepoints':'int16'})
                    .compute()
)

df_percentiles.to_hdf('/home/lag36/scratch/lag36/2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed/Barcodes/percentiles.hdf5',
                       key='data')

In [253]:
df_percentiles = pd.read_hdf('/home/lag36/scratch/lag36/2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed/Barcodes/percentiles.hdf5')

In [352]:
df_bar_fish[df_bar_fish['trenchid'] == 66600]

,trenchid,Barcode Signal,Barcode,consensus,reference,alignmentstart,cigar,subsample,barcode depth,closest_match,...,locus_tag,offset,Gene,sgRNA Sequence,Target Sequence,Category,Predicted_Efficacy,TargetID,barcode,barcodeid
96408,66600,"[3166.12, 2602.04, 2043.06, 4052.1, 2004.02, 1...",110100000000101100100001101110,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1.0,1091M,NaN,696.0,2,...,BSU_33970,1070.0,araR,TTATTCATTCAGTTTTCGTG,CACGAAAACTGAATGAATAA,nonessential,1.0,8806.0,111000010110000010000011000100,81303


In [205]:
df_bar_fish_metadata.keys()

dict_keys(['Total Trenches', 'Total Trenches With Cells', 'Removed Bits', 'Removed Bit Cycles', 'Cycle Order', 'Barcode Length', 'Hamming Threshold', 'Channel Names', 'Summed Intensity Threshold', 'Bit Threshold List'])

In [208]:
# Group by trenchid, and then sample n=10 trenchids
trench_ids = df_percentiles['trenchid'].unique().compute()
# Sample n trenchids
trench_ids = np.random.choice(trench_ids, size=200000, replace=False)

In [293]:
df_sums = df_percentiles.groupby(['trenchid'])[['RFP 98th Percentile', 'Cy5 98th Percentile']].sum().compute().assign(sum_all=lambda df_: df_['RFP 98th Percentile'] + df_['Cy5 98th Percentile'])

In [283]:
a = df_percentiles['trenchid'].unique().compute()

In [310]:
data_sum = hv.Dataset(df_sums[['RFP 98th Percentile', 'sum_all']])
points_sum = data_sum.to(hv.Points, kdims=['RFP 98th Percentile', 'sum_all'])
histogram(points_sum, num_bins=500, bin_range=(70_000,200_000) ,dimension='sum_all')*hv.VLine(95_000).opts(line_width=1, width=1000)

:Overlay
   .Histogram.I :Histogram   [sum_all]   (Count)
   .VLine.I     :VLine   [x,y]

In [257]:
df_bar_kymo_lookup_filt_in_file.head()

,trenchid,File Index,File Trench Index
66600,66600,100,0
66614,66614,100,14
66619,66619,100,19
66632,66632,100,32
66634,66634,100,34


In [209]:
# Get the percentiles for each trenchid
df_percentiles_sample = (df_percentiles
                  .loc[df_percentiles['trenchid'].isin(trench_ids)]
                  .compute()
                  .reset_index()
                  .loc[:,['timepoints', 'trenchid', 'RFP 98th Percentile', 'Cy5 98th Percentile']]
                 )

In [ ]:
data_hists = hv.Dataset(df_percentiles_sample)

bit_thresh_list = df_bar_fish_metadata['Bit Threshold List']

bit_thresholds = pd.DataFrame({
    'timepoints': np.arange(15, dtype=np.int8),
    'RFP 98th Percentile': bit_thresh_list[:15],
    'Cy5 98th Percentile': bit_thresh_list[15:],
})

In [213]:
hv_bit_thresholds = hv.Dataset(bit_thresholds)

In [289]:
from holoviews.operation import histogram

points = data_hists.to(hv.Points, kdims=['RFP 98th Percentile', 'Cy5 98th Percentile'], groupby=['timepoints'])

lines_RFP = hv_bit_thresholds.to(hv.VLines, kdims=['RFP 98th Percentile'], groupby=['timepoints']).opts(line_width=1)
hist_RFP  = histogram(points, num_bins=500, bin_range=(1_800, 5_500), dimension='RFP 98th Percentile').opts(line_width=1,
                                                                                                            # Set log scale on y
                                                                                                            ylim=(0, 1e3), width = 300, height = 150)
lines_Cy5 = hv_bit_thresholds.to(hv.VLines, kdims=['Cy5 98th Percentile'], groupby=['timepoints']).opts(line_width=1)
hist_Cy5  = histogram(points, num_bins=500, bin_range=(2_800, 15_500), dimension='Cy5 98th Percentile').opts(line_width=1,
                                                                                                            # Set log scale on y
                                                                                                            ylim=(0, 1e3), width = 300, height = 150
                                                                                                            )

x = ((hv.NdLayout(hist_RFP*lines_RFP))+(hv.NdLayout(hist_Cy5*lines_Cy5))).opts(
                                                                               # Set single row                                                       
                                                                               shared_axes=False).cols(2)
    # hv.NdLayout(histogram(points, num_bins=500, dimension='Cy5 98th Percentile')) ).opts(shared_axes=False)

In [306]:
import panel

In [310]:
df_bar_kymo_lookup_filt_in_file

,trenchid,File Index,File Trench Index
66600,66600,100,0
66614,66614,100,14
66619,66619,100,19
66632,66632,100,32
66634,66634,100,34
...,...,...,...
67189,67189,100,589
67191,67191,100,591
67192,67192,100,592
67205,67205,100,605


In [357]:
trenchid = 66619
show_bc_kymos(trenchid, show_labels=False)

:Layout
   .Overlay.I   :Overlay
      .Image.I  :Image   [x,y]   (BF)
      .Labels.I :Labels   [x,y]   (timepoints)
   .Overlay.II  :Overlay
      .Image.I  :Image   [x,y]   (RFP)
      .Labels.I :Labels   [x,y]   (timepoints)
   .Overlay.III :Overlay
      .Image.I  :Image   [x,y]   (Cy5)
      .Labels.I :Labels   [x,y]   (timepoints)

In [ ]:
hv_perc_trenchid = hv.Dataset(df_percentiles[df_percentiles['trenchid']==trenchid])

lines_trench_RFP = hv_perc_trenchid.to(hv.VLines, kdims=['RFP 98th Percentile'], groupby=['timepoints']).opts(line_width=1, color='red')
lines_trench_Cy5 = hv_perc_trenchid.to(hv.VLines, kdims=['Cy5 98th Percentile'], groupby=['timepoints']).opts(line_width=1, color='red')

super_layout = panel.Row(
    hv.NdLayout(hist_RFP*lines_RFP*lines_trench_RFP).opts(shared_axes=False).cols(3),
    show_bc_kymos(trenchid, show_labels=False),
    hv.NdLayout(hist_Cy5*lines_Cy5*lines_trench_Cy5).opts(shared_axes=False).cols(3),
)

super_layout

BokehModel(combine_events=True, render_bundle={'docs_json': {'484057db-decf-459e-8b11-9b7fd3cd4ada': {'version…

In [ ]:
# (hv.NdLayout(hist_RFP*lines_RFP*lines_trench_RFP)+hv.NdLayout(hist_Cy5*lines_Cy5*lines_trench_Cy5)).opts(shared_axes=False).cols(2)
(((hv.NdLayout(hist_RFP*lines_RFP*lines_trench_RFP)).opts(shared_axes=False).cols(3) + (hv.NdLayout(hist_Cy5*lines_Cy5*lines_trench_Cy5)).opts(shared_axes=False).cols(3)) + show_bc_kymos(66600, show_labels=False))


:Layout
   .NdLayout.I  :NdLayout   [timepoints]
      :Overlay
         .Histogram.I :Histogram   [RFP 98th Percentile]   (Count)
         .VLines.I    :VLines   [RFP 98th Percentile]
         .VLines.II   :VLines   [RFP 98th Percentile]
   .NdLayout.II :NdLayout   [timepoints]
      :Overlay
         .Histogram.I :Histogram   [Cy5 98th Percentile]   (Count)
         .VLines.I    :VLines   [Cy5 98th Percentile]
         .VLines.II   :VLines   [Cy5 98th Percentile]
   .Image.I     :Image   [x,y]   (BF)
   .Image.II    :Image   [x,y]   (RFP)
   .Image.III   :Image   [x,y]   (Cy5)

#### END OF percentiles!!

In [49]:
bc_kymos_map = {trenchid: show_bc_kymos(trenchid) for trenchid in trenchids_annotated['trenchid'].values[:100]}

NameError: name 'trenchids_annotated' is not defined

In [ ]:
def get_bc_kymos(trenchid):
    return bc_kymos_map[trenchid]


In [ ]:
# Size in MB of bc_kymos_map
import sys
bc_kymos_map_size = sum([sys.getsizeof(bc_kymos_map[trenchid]) for trenchid in bc_kymos_map.keys()]) / (1024*1024)

In [ ]:
from ipywidgets import interact, widgets

In [167]:
# Make a slider for the trenchid across values in df_bar_kymo_lookup_filt_in_file['trenchid'].values[:3]}
trenchid_slider = widgets.SelectionSlider(
    options=df_bar_kymo_lookup_filt_in_file['trenchid'],
    value=df_bar_kymo_lookup_filt_in_file['trenchid'].values[0],
    description='Trenchid',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    style={'description_width': 'initial'},
)

In [169]:
# interact(get_bc_kymos, trenchid=trenchid_slider)
interact(show_bc_kymos, trenchid=trenchid_slider)

interactive(children=(SelectionSlider(continuous_update=False, description='Trenchid', options=(57, 198, 217, …

<function __main__.show_bc_kymos(trenchid)>

In [ ]:
# TODO
#  Make faster by not using where, index directly
#  Later: (LONG) Show with the histograms, and show the next closest barcode
#  Check the assignment pipeline. Ask Daniel about it


#### Show raw images (fov)

In [172]:
df_bar_kymo_meta

,fov,timepoints,lane orientation,y (local),x (local),row,File Index,File Trench Index,trenchid,Trenchid Timepoint Index,t,File Index_fov,Image Index
0,0,0,bottom,18.011834,3.072607,0,0,0,0,0,0.666,0,0
1,0,1,bottom,18.435641,3.284511,0,0,0,0,1,0.456,0,1
2,0,2,bottom,18.223737,3.284511,0,0,0,0,2,0.681,0,2
3,0,3,bottom,18.223737,3.284511,0,0,0,0,3,0.619,0,3
4,0,4,bottom,18.223737,3.284511,0,0,0,0,4,0.618,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20119375,755,10,top,574.047730,622.891584,9,2013,633,1341291,13412910010,4342.032,755,10
20119376,755,11,top,574.259633,623.103488,9,2013,633,1341291,13412910011,4340.741,755,11
20119377,755,12,top,574.259633,623.103488,9,2013,633,1341291,13412910012,4336.821,755,12
20119378,755,13,top,574.259633,622.891584,9,2013,633,1341291,13412910013,4338.600,755,13


In [175]:
headpath_bar_fovs = headpath + '/Barcodes/hdf5/hdf5_'
headpath_phe_fovs = headpath + '/Growth_Division/hdf5/hdf5_'

fov_file_index = 0

data_fov = (xr
        # TODO: Parametrize the file name
        .open_dataset(headpath_bar_fovs+str(fov_file_index)+'.hdf5', phony_dims='sort')
        .swap_dims({'phony_dim_0':'timepoints', 'phony_dim_1':'y', 'phony_dim_2':'x'})
)
data_fov = (data_fov
        .assign_coords({
                        'timepoints': np.arange(data_fov.sizes['timepoints'], dtype=np.int16),
                        'y': np.arange(data_fov.sizes['y'], dtype=np.int16),
                        'x': np.arange(data_fov.sizes['x'], dtype=np.int16),})
)

In [176]:
# Given a trenchid (bc):
bar_trenchid = 57

df_bar_trench = df_bar[df_bar['trenchid'] == bar_trenchid]
df_bar_trench

# Exception if more than one phe_trenchid
assert df_bar_trench['phenotype trenchid'].nunique() == 1
# Get the phenotype trenchid
# phe_trenchid = df_bar_trench.iloc[0]['phenotype trenchid']

df_bar_kymo_meta_trench = df_bar_kymo_meta[df_bar_kymo_meta['trenchid'] == bar_trenchid]

In [177]:
# File indices are different
# df_bar_trench['File Index_fov'].unique()
df_bar_trench.loc[:, ['File Index_fov', 'Image Index']]

df_bar_trench.loc[:, ['File Index_fov', 'Image Index']]

dict_fovs = {}
for file_index in df_bar_trench['File Index_fov'].unique():
    dict_fovs[file_index] = df_bar_trench.loc[df_bar_trench['File Index_fov'] == file_index, 'Image Index'].values

# Open the files
data_phe_fovs = {}
for file_index, image_indices in dict_fovs.items():
    data_phe_fovs[file_index] = (xr
        .open_dataset(headpath_phe_fovs+str(file_index)+'.hdf5', phony_dims='sort')
        .swap_dims({'phony_dim_0':'timepoints', 'phony_dim_1':'y', 'phony_dim_2':'x'})
    )

for key in data_phe_fovs.keys():
    data_phe_fovs[key] = (data_phe_fovs[key]
        .assign_coords({
                        'timepoints': np.arange(data_phe_fovs[key].sizes['timepoints'], dtype=np.int16),
                        'y': np.arange(data_phe_fovs[key].sizes['y'], dtype=np.int16),
                        'x': np.arange(data_phe_fovs[key].sizes['x'], dtype=np.int16),})
    )

# Slice and combine according to dict_fovs
data_phe_fovs = xr.concat([data_phe_fovs[key].isel({'timepoints':image_indices}) for key, image_indices in dict_fovs.items()], dim='timepoints')
data_phe_fovs = data_phe_fovs.assign_coords({'timepoints': np.arange(data_phe_fovs.sizes['timepoints'], dtype=np.int16)})


In [178]:
# Now generate the ROIs
# For a single one:
microns_per_pixel = 0.105951961895293*2
len_y = data_kymo.sizes['y'] # 150
len_x = data_kymo.sizes['x'] # 11

# Covert to an array of tuples
roi = [tuple(row) for row in df_bar_kymo_meta_trench.loc[:,['x (local)', 'y (local)']].values/microns_per_pixel]

## Check lane orientation
# assert that it is the same for all timepoints
assert df_bar_kymo_meta_trench['lane orientation'].nunique() == 1
# Get the lane orientation
lane_orientation = df_bar_kymo_meta_trench.iloc[0]['lane orientation']
if lane_orientation == 'bottom':
    bounds_list = [hv.Bounds((x-len_x/2, y, x+len_x/2, y+len_y)) for x, y in roi]
    roi_bounds = hv.Path(bounds_list)
    roi_bounds = hv.Path(bounds_list[0])
# roi_bounds = hv.Path([hv.Bounds((x-1, y-1, x+0, y+0)) for x, y in roi])


# Generate it for 3 channels: RFP, GFP, and YFP
hvplot_app = {}
for channel in ['BF', 'RFP', 'Cy5']:
    hvplot_app[channel] = data_fov[channel].hvplot.image(
        groupby='timepoints',
        title=channel,
        cmap='gray',
        # clim=(0, 20),
        data_aspect=1,
        # widget_location='bottom',
        rasterize=True,
        # rtol=1e-2,
        # Make it have a single column

        
    ).opts(invert_yaxis=True,
           # Remove axes labels and ticks
              xaxis=None,
              yaxis=None,
              # Remove colorbar
                colorbar=False,
               
           )
# Display the app
# combined = (hvplot_app['BF'] + hvplot_app['Cy5'] + hvplot_app['RFP'])
# combined

In [179]:
hvplot_app['BF']*roi_bounds

BokehModel(combine_events=True, render_bundle={'docs_json': {'b80876fd-e909-420d-bbfd-706994cd1390': {'version…

In [180]:
# Now for the phenotype Roi 
roi_phe = [tuple(row) for row in df_bar_trench.loc[:,['x (local)', 'y (local)']].values/microns_per_pixel]

assert df_bar_trench['lane orientation'].nunique() == 1
lane_orientation_phe = df_bar_trench.iloc[0]['lane orientation']

if lane_orientation_phe == 'bottom':
    bounds_list = [hv.Bounds((x-len_x/2, y, x+len_x/2, y+len_y)) for x, y in roi_phe]
    # roi_bounds_phe = hv.Path(bounds_list)
    roi_bounds_phe = hv.Path(bounds_list[-1]).opts(line_color='red', line_width=1)
roi_bounds_phe


# Figure
hvplot_app_phe = {}
for channel in ['mCherry']:
    hvplot_app_phe[channel] = data_phe_fovs[channel].hvplot.image(
        groupby='timepoints',
        title=channel,
        cmap='kr',
        # clim=(0, 20),
        data_aspect=1,
        # widget_location='bottom',
        rasterize=True,
        # rtol=1e-2,
        # Make it have a single column

        
    ).opts(invert_yaxis=True,
           # Remove axes labels and ticks
              xaxis=None,
              yaxis=None,
              # Remove colorbar
                colorbar=False,
               
           )
# hvplot_app_phe['mCherry']*roi_bounds_phe

In [181]:
hvplot_app_phe['mCherry']

BokehModel(combine_events=True, render_bundle={'docs_json': {'fbdacaef-8353-43f9-ab07-2356809a45bc': {'version…

In [ ]:
hvplot_app['BF']*hvplot_app_phe['mCherry'].apply.opts(alpha = 0.8)*roi_bounds_phe*roi_bounds

In [ ]:
# Display size of df_phe in GB

# Starting from a phenotype kymo (with its associated trenchid), I need to:
# - Get the position on the phe fov
# - Get the corresponding barcode trenchid
# - Get the corresponding barcode fov
# - Get the corresponding barcode kymo
# - Match the positions in the fov images



#### Kymo and label an entire file

#### Find and scroll through kymographs

##### TODO
- Make modular

In [ ]:
# Say I start with a trenchid from the phenotype data.

# I find it by looking at kymographs, so write a function to get kymographs and scroll through them
import h5py
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

headpath 

def plot_kymograph(indices, channels, trench_index, kymo_index, headpath, subfolder = "kymograph", prefix = "kymograph", cmap = "gray"):
    file_path = headpath + subfolder + "/" + prefix+"_" + str(kymo_index) + ".hdf5"
    # print(file_path)
    with h5py.File(file_path, "r") as infile:
        # print(infile['mCherry'].shape)
        # indices = list(range(0, 1, 1))

        for ch in channels:
            img = infile[ch][trench_index][indices[0]]
            for ind in indices[1:]:
                img_ind = infile[ch][trench_index][ind]
                img = np.concatenate((img, img_ind), axis=1)
            _= plt.figure(figsize = (20, 20))
            _ = plt.imshow(img, cmap=cmap, aspect=1)
            # Annotate kymo and trench index
            _= plt.text(10, 10, f"Kymo: {kymo_index}, Trench: {trench_index}", color="w")
            _=plt.show()
            _=plt.close()

indices = np.arange(0, 67)
# Create a function to update the plot
def update_plot(trench_index, kymo_index):
    # y = np.sin(frequency * x)
    _ = plt.clf()  # Clear the previous plot
    channels = ["mCherry"]
    plot_kymograph(indices, channels, trench_index, kymo_index, headpath, subfolder = "Growth_Division/kymograph", prefix = "kymograph", cmap='gray')
    _ = plt.show()
    # channels = ["data"]
    # plot_kymograph(indices, channels, trench_index, kymo_index, headpath, subfolder = "fluorsegmentation", prefix="segmentation", cmap="jet")
    # _ = plt.show()

# Create an interactive slider widget
trench_index_slider = widgets.IntSlider(min=0, max=149, step=1, value=0)
kymo_index_slider = widgets.IntSlider(min=0, max=500, step=1, value=0)

In [ ]:
# Use the interact function to connect the slider to the update_plot function
interact(update_plot, trench_index=trench_index_slider, kymo_index=kymo_index_slider)# kymo_index=kymo_index_slider)

In [ ]:
df_lookup_bar = df_lookup = (df_bar
             .loc[:,['trenchid', 'Barcode Signal', 'Barcode', 'closest_match',  'barcode depth', 'Closest Hamming Distance', 'Category', 'barcode', 'barcodeid']]
             .merge(df_bar_kymo_meta.groupby('trenchid').first().reset_index(),
                    left_on='trenchid',
                    right_on='trenchid', how='left')
                )

In [ ]:

bc_kymo_metadata_df = pd.read_hdf(headpath+"/kymograph_metadata_df.hdf5", key='data')
hdf5_metadata_df = pd.read_hdf("/home/lag36/scratch/lag36/2024-12-12_lLAG8_EZRDM_Run-02_3-Fiducials_Auto-Media-Switch/Barcodes/metadata.hdf5")
# df_2_1
df_lookup = (df_2_1
             .loc[:,['trenchid', 'Barcode Signal', 'Barcode', 'closest_match',  'barcode depth', 'Closest Hamming Distance', 'Category', 'barcode', 'barcodeid']]
             .merge(bc_kymo_metadata_df.groupby('trenchid').first().reset_index(),
                    left_on='trenchid',
                    right_on='trenchid', how='left')
                )